# DiT Next-Frame Prediction Evaluation

Loads a trained DiT checkpoint, runs `rollout()` on a DROID val sample with GT actions,
and visualizes predicted vs GT next frames.

Two visualization modes:
- **SH→RGB**: Direct DC-band SH extraction (`rgb = 0.5 + C0 * sh`)
- **Rendered**: Full Gaussian Splatting render via `diff-gaussian-rasterization`

In [15]:
import os, sys
sys.path.insert(0, os.path.expandvars("$GWM_PATH"))
import torch, numpy as np, matplotlib.pyplot as plt
from hydra import compose, initialize
from hydra.core.global_hydra import GlobalHydra

In [16]:
GWM_PATH = os.environ["GWM_PATH"]

GlobalHydra.instance().clear()
with initialize(config_path="../configs", version_base=None):
    cfg = compose(config_name="train_gwm", overrides=[
        "world_model.observation.use_gs=true",
        "world_model.reward.use_reward_model=false",
        "world_model.vae.use_vae=false",
        f"dataset.data_path={GWM_PATH}/data/",
    ])

print("Config loaded. context_length:", cfg.world_model.context_length)

Config loaded. context_length: 2


In [17]:
from gaussianwm.gwm_predictor import GaussianPredictor

CKPT = f"{GWM_PATH}/logs/gwm/checkpoints/model_latest.pt"
model = GaussianPredictor(cfg.world_model).cuda()
state_dict = torch.load(CKPT, map_location="cuda")
model.model.load_state_dict(state_dict)
model.eval()
print(f"Loaded DiT from {CKPT}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]
Loading model from: /home/frankcholula/Workspace/gaussianwm/.venv/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
🔥🔥🔥 Splatt3r Model loaded from /home/frankcholula/Workspace/gaussianwm/third_party/splatt3r/checkpoints/splatt3r_v1.0/epoch=19-step=1200.ckpt
[Model] Trainable parameters: 33.205088M
[Model] Total parameters: 33.598304M
loaded pretrained LPIPS loss from /home/frankcholula/Workspace/gaussianwm/ckpt/lpips/vgg.pth
Loaded DiT from /home/frankcholula/Workspace/gaussianwm/logs/gwm/checkpoints/model_latest.pt


/tmp/ipykernel_2745812/3558710270.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(CKPT, map_location="cuda")


In [18]:
import tensorflow_datasets as tfds
import tensorflow as tf
import glob, json
from gaussianwm.processor.datasets import euler_to_rmat, mat_to_rot6d
from PIL import Image as PILImage

GWM_PATH = os.environ["GWM_PATH"]

# Load episode 0 directly — no RLDS pipeline, no threads, no shuffle buffer
ds    = tfds.load("droid_100", data_dir=f"{GWM_PATH}/data/", split="train", with_info=False)
ep    = next(iter(ds))
steps = list(ep["steps"])
print(f"Episode 0: {len(steps)} frames  ({len(steps)/15:.1f}s @ 15Hz)")

# Extract raw images (resize to 128×128) and action components
frames, raw_cart, raw_grip = [], [], []
for step in steps:
    img = PILImage.fromarray(step["observation"]["exterior_image_1_left"].numpy())
    frames.append(np.array(img.resize((128, 128), PILImage.BILINEAR)))
    raw_cart.append(step["action_dict"]["cartesian_position"].numpy())
    raw_grip.append(step["action_dict"]["gripper_position"].numpy())

# Build 10-dim action: xyz(3) + rot6d(6) + gripper(1)  — same as droid_dataset_transform
cart_tf = tf.constant(np.stack(raw_cart), dtype=tf.float32)
grip_tf = tf.constant(np.stack(raw_grip), dtype=tf.float32)
actions_10d = tf.concat([
    cart_tf[:, :3],
    mat_to_rot6d(euler_to_rmat(cart_tf[:, 3:6])),
    grip_tf,
], axis=-1).numpy().astype(np.float32)   # [T, 10]

# Bounds normalization (q01/q99) — same as training pipeline
stats_file = sorted(glob.glob(f"{GWM_PATH}/data/droid_100/1.0.0/dataset_statistics_*.json"))[0]
with open(stats_file) as f:
    stats = json.load(f)
q01 = np.array(stats["action"]["q01"], dtype=np.float32)
q99 = np.array(stats["action"]["q99"], dtype=np.float32)
actions_norm = 2 * (actions_10d - q01) / (q99 - q01 + 1e-8) - 1
actions_norm[:, 9] = actions_10d[:, 9]   # gripper dim: not normalized

# Match pipeline shift: trajectory["action"][1:]
actions_norm = actions_norm[1:]

obs    = torch.from_numpy(np.stack(frames))        # [T,   128, 128, 3] uint8
action = torch.from_numpy(actions_norm)            # [T-1, 10]  float32
reward = torch.zeros(len(steps) - 1)

print(f"obs: {obs.shape},  action: {action.shape}")

Episode 0: 166 frames  (11.1s @ 15Hz)
obs: torch.Size([166, 128, 128, 3]),  action: torch.Size([165, 10])


In [19]:
import imageio
from PIL import Image, ImageDraw

# ── Save ground truth frames as gt.gif ────────────────────────────────────────
# Run this cell to preview the selected clip BEFORE the expensive DiT rollout.
# Each frame is labelled with its timestamp in seconds.

GT_GIF_FPS   = 5     # slow playback so motion is easy to see
GT_GIF_SCALE = 3     # 3× upscale: 128px → 384px
GT_GIF_OUT   = "gt.gif"

T = obs.shape[0]
gt_frames = []

for t in range(T):
    frame_u8 = obs[t].numpy()   # [H, W, 3] uint8

    img = Image.fromarray(frame_u8).resize(
        (frame_u8.shape[1] * GT_GIF_SCALE, frame_u8.shape[0] * GT_GIF_SCALE),
        Image.NEAREST,
    )
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, img.width, 16], fill=(0, 0, 0))
    draw.text((4, 2), f"GT  frame={t}  t={t/15:.2f}s", fill=(255, 255, 255))
    gt_frames.append(np.array(img))

imageio.mimsave(GT_GIF_OUT, gt_frames, fps=GT_GIF_FPS, loop=0)
print(f"Saved {GT_GIF_OUT}  —  {T} frames @ {GT_GIF_FPS}fps  "
      f"({T/GT_GIF_FPS:.1f}s playback,  {T/15:.1f}s real time)")

Saved gt.gif  —  166 frames @ 5fps  (33.2s playback,  11.1s real time)


In [ ]:
# The clip+quantize fix is now applied in source:
# DenoiserConfig(quantize_output=not use_gs) in gwm_predictor.py
# → wrap_model_output passes Gaussian features through unmodified when use_gs=True.
# No monkey-patching needed.

In [ ]:
CONTEXT     = cfg.world_model.context_length   # 2
MAX_ROLLOUT = 8    # match training horizon (rollout_horizon=8 in train_gwm.yaml)
START_FRAME = 25   # start context here; movement begins ~frame 20

HORIZON = min(
    obs.shape[0]    - START_FRAME - CONTEXT,
    action.shape[0] - START_FRAME - CONTEXT,
    MAX_ROLLOUT,
)
print(f"obs frames: {obs.shape[0]}, action steps: {action.shape[0]}")
print(f"START_FRAME={START_FRAME}, CONTEXT={CONTEXT}, HORIZON={HORIZON}")
print(f"Predicting frames {START_FRAME+CONTEXT} – {START_FRAME+CONTEXT+HORIZON-1}  "
      f"({(START_FRAME+CONTEXT)/15:.2f}s – {(START_FRAME+CONTEXT+HORIZON-1)/15:.2f}s)")

context   = obs[START_FRAME:START_FRAME+CONTEXT].permute(0, 3, 1, 2).float()
obs_input = context.reshape(1, -1, 128, 128).cuda()

gt_actions = action[START_FRAME+CONTEXT : START_FRAME+CONTEXT+HORIZON].float().cuda()

def gt_policy(obs_t, t):
    return gt_actions[t].unsqueeze(0)

with torch.no_grad():
    rollout_obs, rollout_actions, rollout_rewards = model.rollout(
        obs_input, policy=gt_policy, horizon=HORIZON
    )
print(f"rollout_obs: {rollout_obs.shape}")
assert rollout_obs.shape == (1, HORIZON + 1, 28, 128, 128)

In [7]:
# Wire up Gaussian Splatting renderer from the already-loaded Splatt3r submodule.
# regressor.py already appended these paths when GaussianPredictor was initialised,
# so the imports below should resolve immediately (modules are already in sys.modules).
from src.pixelsplat_src.cuda_splatting import render_cuda
from utils.geometry import build_covariance
from utils.geometry import normalize_intrinsics as _norm_K

H, W = 128, 128

# Standard pinhole intrinsics in pixel space: focal = image_width → ≈53° FoV
# After normalize_intrinsics: [[1, 0, 0.5], [0, 1, 0.5], [0, 0, 1]]
_K_px  = torch.tensor([[[float(W), 0., W/2.],
                          [0., float(H), H/2.],
                          [0.,      0.,    1.]]],
                        dtype=torch.float32, device="cuda")
_K_norm = _norm_K(_K_px, (H, W))           # [1, 3, 3]

# Identity camera-to-world (c2w=I): camera at origin, looking along +Z
# Gaussian means are in view1's camera frame → identity c2w renders view1's image
_C2W  = torch.eye(4, dtype=torch.float32, device="cuda").unsqueeze(0)  # [1, 4, 4]
_near = torch.tensor([0.1],   dtype=torch.float32, device="cuda")
_far  = torch.tensor([100.0], dtype=torch.float32, device="cuda")
_bg   = torch.zeros(1, 3,    dtype=torch.float32, device="cuda")


def render_gaussians(gauss_14hw: torch.Tensor) -> np.ndarray:
    """Render a [14, H, W] CUDA tensor → [H, W, 3] numpy RGB in [0, 1].

    14-dim layout (must match Splatt3r / GWM convention):
      0:3  → XYZ means  (in view1 camera frame)
      3:6  → scales     (linear, post-activation)
      6:10 → rotations  (quaternion xyzw)
     10:13 → SH DC band (3 colour channels, 1 coefficient each)
     13:14 → opacity    (sigmoid-activated, clamped to [0, 1])
    """
    N = H * W
    g = gauss_14hw.reshape(14, N).T.contiguous()   # [N, 14]

    means  = g[:, 0:3]                             # [N, 3]
    scales = g[:, 3:6]                             # [N, 3]
    rots   = g[:, 6:10]                            # [N, 4]  xyzw
    sh_dc  = g[:, 10:13]                           # [N, 3]
    opc    = g[:, 13:14].clamp(0, 1)               # [N, 1]

    covs = build_covariance(scales, rots)          # [N, 3, 3]

    rendered = render_cuda(
        _C2W,                              # [1, 4, 4]
        _K_norm,                           # [1, 3, 3]
        _near,                             # [1]
        _far,                              # [1]
        (H, W),
        _bg,                               # [1, 3]
        means.unsqueeze(0),                # [1, N, 3]
        covs.unsqueeze(0),                 # [1, N, 3, 3]
        sh_dc.unsqueeze(-1).unsqueeze(0),  # [1, N, 3, 1]  (degree-0 SH)
        opc.squeeze(-1).unsqueeze(0),      # [1, N]
    )  # → [1, 3, H, W]

    # render_cuda sets requires_grad=True internally (mean_gradients for rasterizer)
    return rendered[0].permute(1, 2, 0).clamp(0, 1).detach().cpu().numpy()  # [H, W, 3]


# ── Baseline SH→RGB helpers (no rendering) ────────────────────────────────────
C0 = 0.28209479177387814

def gauss_to_rgb(gauss_14hw: torch.Tensor) -> np.ndarray:
    """[14, H, W] → [H, W, 3] numpy via direct DC-band SH→RGB (fast baseline)."""
    sh = gauss_14hw[10:13]  # [3, H, W]
    return (0.5 + C0 * sh).clamp(0, 1).permute(1, 2, 0).detach().cpu().numpy()

def gt_frame_rgb(obs_thwc: torch.Tensor, t: int) -> np.ndarray:
    """Raw uint8 GT frame at time t → [H, W, 3] numpy in [0, 1]."""
    return obs_thwc[t].float().numpy() / 255.


print("Renderer ready  (identity c2w, focal=image_width)")
print("SH→RGB helpers ready  (gauss_to_rgb, gt_frame_rgb)")

Renderer ready  (identity c2w, focal=image_width)
SH→RGB helpers ready  (gauss_to_rgb, gt_frame_rgb)


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

psnrs_sh,     ssims_sh     = [], []
psnrs_render, ssims_render = [], []

for t in range(HORIZON):
    gt_t   = START_FRAME + CONTEXT + t
    gt_img = gt_frame_rgb(obs, gt_t)

    pred_gauss = rollout_obs[0, t+1, 14:].clone()

    pred_sh = gauss_to_rgb(pred_gauss)
    p_sh = psnr(gt_img, pred_sh, data_range=1.0)
    s_sh = ssim(gt_img, pred_sh, data_range=1.0, channel_axis=2)
    psnrs_sh.append(p_sh);  ssims_sh.append(s_sh)

    pred_rend = render_gaussians(pred_gauss)
    p_r = psnr(gt_img, pred_rend, data_range=1.0)
    s_r = ssim(gt_img, pred_rend, data_range=1.0, channel_axis=2)
    psnrs_render.append(p_r);  ssims_render.append(s_r)

    print(f"t={gt_t:2d}  SH→RGB: PSNR={p_sh:.2f} dB  SSIM={s_sh:.4f}"
          f"   |   Rendered: PSNR={p_r:.2f} dB  SSIM={s_r:.4f}")

print(f"\nMean SH→RGB  :  PSNR={np.mean(psnrs_sh):.2f} dB   SSIM={np.mean(ssims_sh):.4f}")
print(f"Mean Rendered:  PSNR={np.mean(psnrs_render):.2f} dB   SSIM={np.mean(ssims_render):.4f}")

In [ ]:
# ── Static Scene Diagnostics ──────────────────────────────────────────────────
# Three tests:
#   1. Copy-forward baseline: does DiT beat simply repeating the last context frame?
#   2. Inter-frame pixel change: how much do consecutive predicted frames differ vs GT?
#   3. Pred vs context: how far does each predicted frame drift from the starting frame?

context_last = gt_frame_rgb(obs, START_FRAME + CONTEXT - 1)   # [H, W, 3] last context frame

# Collect predicted RGB frames once
pred_frames = [gauss_to_rgb(rollout_obs[0, t+1, 14:].clone()) for t in range(HORIZON)]
gt_frames_  = [gt_frame_rgb(obs, START_FRAME + CONTEXT + t)   for t in range(HORIZON)]

# ── Test 1: copy-forward baseline ────────────────────────────────────────────
from skimage.metrics import peak_signal_noise_ratio as psnr
copy_psnrs = [psnr(gt_frames_[t], context_last, data_range=1.0) for t in range(HORIZON)]

print("── Test 1: Copy-forward baseline vs DiT (SH→RGB PSNR) ───────")
print(f"{'frame':>7}  {'copy PSNR':>10}  {'DiT PSNR':>10}  {'DiT - copy':>10}")
for t in range(HORIZON):
    delta = psnrs_sh[t] - copy_psnrs[t]
    flag  = " ✓" if delta > 0 else " ✗ model WORSE than copy"
    print(f"  t={START_FRAME+CONTEXT+t:2d}   {copy_psnrs[t]:>10.2f}  {psnrs_sh[t]:>10.2f}  {delta:>+10.2f} dB{flag}")
print(f"\n  Mean copy: {np.mean(copy_psnrs):.2f} dB  |  Mean DiT: {np.mean(psnrs_sh):.2f} dB  "
      f"|  Δ = {np.mean(psnrs_sh)-np.mean(copy_psnrs):+.2f} dB")

# ── Test 2: inter-frame pixel change ─────────────────────────────────────────
pred_diffs = [np.abs(pred_frames[t+1] - pred_frames[t]).mean() for t in range(HORIZON-1)]
gt_diffs   = [np.abs(gt_frames_[t+1]  - gt_frames_[t] ).mean() for t in range(HORIZON-1)]

print("\n── Test 2: Inter-frame pixel change (mean |I[t+1]−I[t]|, 0–1 scale) ─")
print(f"{'step':>10}  {'pred Δ':>10}  {'GT Δ':>10}  {'pred/GT':>8}")
for t in range(HORIZON-1):
    ratio = pred_diffs[t] / (gt_diffs[t] + 1e-8)
    print(f"  {START_FRAME+CONTEXT+t}→{START_FRAME+CONTEXT+t+1}   {pred_diffs[t]:>10.5f}  {gt_diffs[t]:>10.5f}  {ratio:>8.3f}x")
mean_ratio = np.mean(pred_diffs) / (np.mean(gt_diffs) + 1e-8)
print(f"\n  Mean pred Δ: {np.mean(pred_diffs):.5f}  |  Mean GT Δ: {np.mean(gt_diffs):.5f}  "
      f"|  ratio: {mean_ratio:.3f}x")

# ── Test 3: drift from context frame ─────────────────────────────────────────
pred_vs_ctx = [np.abs(pred_frames[t] - context_last).mean() for t in range(HORIZON)]
gt_vs_ctx   = [np.abs(gt_frames_[t]  - context_last).mean() for t in range(HORIZON)]

print("\n── Test 3: Drift from last context frame (mean |I[t]−I_ctx|) ────────")
print(f"{'frame':>7}  {'pred drift':>12}  {'GT drift':>12}")
for t in range(HORIZON):
    print(f"  t={START_FRAME+CONTEXT+t:2d}   {pred_vs_ctx[t]:>12.5f}  {gt_vs_ctx[t]:>12.5f}")

# ── Verdict ───────────────────────────────────────────────────────────────────
print("\n── Verdict ──────────────────────────────────────────────────────────")
if mean_ratio < 0.05:
    print("STATIC: predicted frames are essentially frozen (pred motion < 5% of GT)")
elif mean_ratio < 0.3:
    print("MOSTLY STATIC: pred changes much less than GT")
else:
    print("DYNAMIC: pred changes comparably to GT")

In [ ]:
N_COLS   = min(HORIZON, 10)
step     = max(1, HORIZON // N_COLS)
display_steps = list(range(0, HORIZON, step))[:N_COLS]

fig, axes = plt.subplots(2, N_COLS, figsize=(N_COLS * 2, 4))

for col, t in enumerate(display_steps):
    gt_t       = START_FRAME + CONTEXT + t
    pred_gauss = rollout_obs[0, t+1, 14:].clone()
    gt_img     = gt_frame_rgb(obs, gt_t)
    pred_sh    = gauss_to_rgb(pred_gauss)

    axes[0, col].imshow(gt_img)
    axes[0, col].set_title(f"GT t={gt_t}\n({gt_t/15:.1f}s)", fontsize=7)
    axes[0, col].axis("off")

    axes[1, col].imshow(pred_sh)
    axes[1, col].set_title(f"Pred\n{psnrs_sh[t]:.1f}dB", fontsize=7)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("GT",   fontsize=9)
axes[1, 0].set_ylabel("Pred", fontsize=9)
plt.suptitle(f"GWM DiT Rollout  (start={START_FRAME}, horizon={HORIZON})  — SH→RGB", fontsize=9)
plt.tight_layout()
plt.savefig("eval_rollout_rendered.png", dpi=150)
plt.show()

In [ ]:
import imageio
from PIL import Image, ImageDraw, ImageFont

GIF_FPS   = 5
GIF_SCALE = 2
OUT_GIF   = "eval_rollout.gif"

frames_gif = []
for t in range(HORIZON):
    gt_t       = START_FRAME + CONTEXT + t
    gt_img     = gt_frame_rgb(obs, gt_t)
    pred_gauss = rollout_obs[0, t+1, 14:].clone()
    pred_sh    = gauss_to_rgb(pred_gauss)

    gt_u8   = (gt_img  * 255).clip(0, 255).astype(np.uint8)
    pred_u8 = (pred_sh * 255).clip(0, 255).astype(np.uint8)

    divider  = np.ones((128, 2, 3), dtype=np.uint8) * 200
    combined = np.concatenate([gt_u8, divider, pred_u8], axis=1)

    img = Image.fromarray(combined).resize(
        (combined.shape[1] * GIF_SCALE, 128 * GIF_SCALE), Image.NEAREST
    )
    draw  = ImageDraw.Draw(img)
    W_sc  = 128 * GIF_SCALE
    draw.rectangle([0,      0, W_sc,       14], fill=(0, 0, 0))
    draw.rectangle([W_sc+2, 0, img.width,  14], fill=(0, 0, 0))
    draw.text((2,        1), f"GT  {gt_t/15:.2f}s",         fill=(255, 255, 255))
    draw.text((W_sc + 4, 1), f"Pred  {psnrs_sh[t]:.1f}dB",  fill=(200, 255, 200))

    frames_gif.append(np.array(img))

imageio.mimsave(OUT_GIF, frames_gif, fps=GIF_FPS, loop=0)
print(f"Saved {OUT_GIF}")

In [ ]:
dim_groups = {
    "XYZ":       (0,  3),
    "Scales":    (3,  6),
    "Rotations": (6,  10),
    "SH/Color":  (10, 13),
    "Opacity":   (13, 14),
}

gt_rgb_tensor = (
    obs[START_FRAME+CONTEXT : START_FRAME+CONTEXT+HORIZON]
    .permute(0, 3, 1, 2)
    .float()
    .unsqueeze(0)
    .cuda()
)

with torch.no_grad():
    gt_gaussians = model._process_obs(gt_rgb_tensor / 255.)

pred_gaussians = rollout_obs[0, 1:, 14:]

print(f"{'Property':<12}  MSE")
print("-" * 25)
for name, (s, e) in dim_groups.items():
    mse = ((pred_gaussians[:, s:e] - gt_gaussians[0, :, s:e]) ** 2).mean().item()
    print(f"{name:<12}: {mse:.6f}")

In [11]:
# ── Splat Position Movement Diagnostic ────────────────────────────────────────
# Does the DiT actually move Gaussian XYZ positions between timesteps,
# or does it predict a static scene by copying the input?

# XYZ slices: rollout dims 14:17 = view1 XYZ (0:3 of the 14-dim layout)
pred_xyz = rollout_obs[0, 1:, 14:17].detach().cpu()   # [8, 3, H, W]  predicted
gt_xyz   = gt_gaussians[0, :,   0:3].detach().cpu()   # [8, 3, H, W]  GT (Splatt3r on GT frames)
init_xyz = rollout_obs[0,  0, 14:17].detach().cpu()   # [3, H, W]     last context frame XYZ

# -- Consecutive frame-to-frame displacement ----------------------------------
print("Consecutive XYZ displacement (mean L2 per splat)")
print(f"{'Step':>8}  {'Pred Δ':>10}  {'GT Δ':>10}  {'Ratio':>8}")
print("-" * 44)
for t in range(HORIZON - 1):
    d_pred = (pred_xyz[t+1] - pred_xyz[t]).norm(dim=0).mean().item()
    d_gt   = (gt_xyz[t+1]   - gt_xyz[t]  ).norm(dim=0).mean().item()
    ratio  = d_pred / (d_gt + 1e-8)
    print(f"t={t+2}→{t+3}   {d_pred:>10.5f}  {d_gt:>10.5f}  {ratio:>8.3f}x")

# -- Cumulative drift from initial context ------------------------------------
print(f"\nCumulative XYZ displacement from initial context frame")
print(f"{'Step':>5}  {'Pred':>10}  {'GT':>10}  {'Ratio':>8}")
print("-" * 40)
for t in range(HORIZON):
    d_pred = (pred_xyz[t] - init_xyz).norm(dim=0).mean().item()
    d_gt   = (gt_xyz[t]   - init_xyz).norm(dim=0).mean().item()
    ratio  = d_pred / (d_gt + 1e-8)
    print(f"t={t+2:>2}   {d_pred:>10.5f}  {d_gt:>10.5f}  {ratio:>8.3f}x")

# -- Overall verdict ----------------------------------------------------------
mean_pred = (pred_xyz - init_xyz.unsqueeze(0)).norm(dim=1).mean().item()
mean_gt   = (gt_xyz   - init_xyz.unsqueeze(0)).norm(dim=1).mean().item()
overall_ratio = mean_pred / (mean_gt + 1e-8)
print(f"\nOverall mean displacement (pred vs init): {mean_pred:.5f}")
print(f"Overall mean displacement (GT   vs init): {mean_gt:.5f}")
print(f"Pred/GT ratio: {overall_ratio:.3f}", end="  ")
if overall_ratio < 0.1:
    print("→ model predicts STATIC scene (near-copy of input)")
elif overall_ratio < 0.5:
    print("→ model moves splats, but significantly UNDERESTIMATES motion")
else:
    print("→ model moves splats comparably to GT")

Consecutive XYZ displacement (mean L2 per splat)
    Step      Pred Δ        GT Δ     Ratio
--------------------------------------------
t=2→3      0.17894     0.00000  17893734.574x
t=3→4      0.14633     0.00000  14633145.928x
t=4→5      0.09740     0.00000  9740404.040x
t=5→6      0.08264     0.00000  8263578.266x
t=6→7      0.05529     0.00000  5528689.548x
t=7→8      0.04085     0.00000  4084721.580x
t=8→9      0.02147     0.00000  2147150.226x

Cumulative XYZ displacement from initial context frame
 Step        Pred          GT     Ratio
----------------------------------------
t= 2      0.45439     0.00795    57.181x
t= 3      0.63323     0.00795    79.686x
t= 4      0.77935     0.00795    98.074x
t= 5      0.87668     0.00795   110.322x
t= 6      0.95922     0.00795   120.709x
t= 7      1.01440     0.00795   127.653x
t= 8      1.05508     0.00795   132.772x
t= 9      1.07622     0.00795   135.433x

Overall mean displacement (pred vs init): 0.85607
Overall mean displacement (GT 